In [1]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt 

In [2]:
def region_of_interest_photo(image):
    height = image.shape[0]
    width = image.shape[1]

    # Define a triangular region of interest
    imshape = image.shape
    polygons = np.array([[(100, height), (width // 2, height // 2), (width - 100, height)]], dtype=np.int32)
    #polygons = np.array([[(0,imshape[0]),(4*imshape[1]/9, 6*imshape[0]/10), (5*imshape[1]/9, 6*imshape[0]/10), (imshape[1],imshape[0])]], dtype=np.int32)
    # Create a mask with the same dimensions as the image
    mask = np.zeros_like(image)
    
    # Fill the mask with the polygon
    cv.fillPoly(mask, polygons, 255)
    
    # Apply the mask to the image
    masked_image = cv.bitwise_and(image, mask)
    return masked_image

In [3]:
def color_mask(image):
    ### create a color mask ###
    #convert from RGB to HSV
    hls = cv.cvtColor(image, cv.COLOR_BGR2HLS)

    #define two color masks for yellow and white
    #white mask is applied in RGB color space, it's easier to tune the values
    #values from 200 to 255 on all colors are picked from trial and error
    mask_white = cv.inRange(hls, (0, 140, 0), (180, 255, 80))

    #yellow mask is done in HSV color space since it's easier to identify the hue of a certain color
    #values from 15 to 25 for hue and above 60 for saturation are picked from trial and error
    mask_yellow = cv.inRange(hls, (15, 30, 80), (45, 204, 255))

    #combine the two masks, both yellow and white pixels are of interest
    color_mask = cv.bitwise_or(mask_white, mask_yellow)

    #make a copy of the original image
    masked_img = np.copy(image)
    
    #pixels that are not part of the mask(neither white or yellow) are made black
    masked_img[color_mask == 0] = [0,0,0]
    return masked_img

In [4]:
def Canny(image, low_threshold=50, high_threshold=150):
    # Convert the image to grayscale
    gray = cv.cvtColor(image, cv.COLOR_RGB2GRAY)
    
    # Apply Gaussian blur to reduce noise
    
    blurred = cv.GaussianBlur(gray, (5, 5), 0)

    # Apply Canny edge detection
    edges = cv.Canny(blurred, low_threshold, high_threshold, apertureSize=3)

    return edges

In [5]:
def hough_lines(image):
    # Apply Hough Line Transform to detect lines in the image
    lines = cv.HoughLinesP(image, 1, np.pi / 180, threshold=20, minLineLength=10, maxLineGap=5)
    
    return lines

In [6]:
def display_lines(image, lines):
    line_image = np.zeros_like(image)
    
    average_lines = merging_lines(image, lines)



    if average_lines is not None:
        for x1, y1, x2, y2 in average_lines:
            cv.line(line_image, (x1, y1), (x2, y2), (255, 0, 0), thickness=10)
    return line_image

    # #display the lines on the original image
    # if lines is not None:
    #     for line in lines:
    #         x1, y1, x2, y2 = line[0]
    #         cv.line(line_image, (x1, y1), (x2, y2), (255, 0, 0), thickness=10)
    # return line_image

In [11]:
from project import make_coordinates


def merging_lines(image, lines, slope_threshold=0.1, x_bottom_threshold=50):
    if lines is None: return None
    height = image.shape[0]
    
    # Container for the original clustering logic
    clusters = [] 

    for line in lines:
        x1, y1, x2, y2 = line.reshape(4)
        if abs(x2 - x1) < 1e-3: continue  # Skip vertical
        
        slope = (y2 - y1) / (x2 - x1)
        intercept = y1 - slope * x1
        
        # 1. Slope filter (ignore horizontal-ish noise)
        if abs(slope) < 0.: continue             

        # 2. Calculate stable bottom-x
        x_bottom = (height - intercept) / slope if slope != 0 else x1

        found_cluster = False
        for cluster in clusters:
            avg_slope, _, avg_xb = np.mean(cluster, axis=0)
            if abs(slope - avg_slope) < slope_threshold and \
                abs(x_bottom - avg_xb) < x_bottom_threshold:
                cluster.append((slope, intercept, x_bottom))
                found_cluster = True
                break
        if not found_cluster:
            clusters.append([(slope, intercept, x_bottom)])

    averaged_lines = []

    for cluster in clusters:
        if len(cluster) > 1: 
            lane_avg = np.mean(cluster, axis=0)
            averaged_lines.append(make_coordinates(image, lane_avg))
            
    return np.array(averaged_lines) 

In [8]:
def make_coordinates(image, line_parameters):    
    slope, intercept, _ = line_parameters  # updated: only 2 values

    height, width = image.shape[:2]

    y1 = height
    y2 = int(height * 0.6)

    # Handle invalid slope
    if slope is None or abs(slope) < 1e-3:
        return None

    # Compute x coordinates
    x1 = (y1 - intercept) / slope
    x2 = (y2 - intercept) / slope

    # Clamp values to image boundaries (VERY important)
    x1 = int(np.clip(x1, 0, width))
    x2 = int(np.clip(x2, 0, width))

    return np.array([x1, y1, x2, y2])

In [9]:
# apply CLAHE (Contrast Limited Adaptive Histogram Equalization)
def apply_clahe(image):
    # Convert the image to LAB color space
    lab = cv.cvtColor(image, cv.COLOR_RGB2LAB)

    # Split the LAB image into its channels
    l, a, b = cv.split(lab)

    # Apply CLAHE to the L channel
    clahe = cv.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)

    # Merge the CLAHE enhanced L channel back with A and B channels
    limg = cv.merge((cl, a, b))

    # Convert back to RGB color space
    final_image = cv.cvtColor(limg, cv.COLOR_LAB2RGB)

    return final_image

In [ ]:
# testing the code on a video
cap = cv.VideoCapture('roads/video2.mp4')
frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    uniform_frame = apply_clahe(frame)
    masked_frame = color_mask(uniform_frame)
    canny_frame = Canny(masked_frame, 50, 150)
    cropped_frame = region_of_interest_photo(canny_frame)
    lines_frame = hough_lines(cropped_frame)
    line_image_frame = display_lines(frame, lines_frame)
    combo_image_frame = cv.addWeighted(frame, 0.8, line_image_frame, 1, 1)

    cv.imshow('Lane Detection', cropped_frame)

    if cv.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv.destroyAllWindows()

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (27,) + inhomogeneous part.

: 

In [10]:
# looping through all the sample road s and a qpplying the pipeline to each of them
for i in range(1, 5):
    road = cv.imread(f'roads/road{i}.jpg')
    if road is None:
        road = cv.imread(f'roads/road{i}.png')
        if road is None:
            print(f"Error loading image roads/road{i}.jpg or roads/road{i}.png")
            continue
    uniform_road = apply_clahe(road)
    masked_road = color_mask(uniform_road)
    canny_image = Canny(masked_road,50, 150)
    cropped_image = region_of_interest_photo(canny_image)
    lines = hough_lines(cropped_image)
    line_image = display_lines(road, lines)
    combo_image = cv.addWeighted(road, 0.8, line_image, 1, 1)
    cv.imshow(f'Masked Road {i}', combo_image)
    cv.waitKey(5000)
    cv.destroyAllWindows()

[ WARN:0@21.794] global loadsave.cpp:278 findDecoder imread_('roads/road1.jpg'): can't open/read file: check file path/integrity


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (78,) + inhomogeneous part.